In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import HistGradientBoostingRegressor, StackingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import BaggingRegressor
import time

In [7]:
# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# lists previously defined in 05_0_preproc_helpers.ipynb
numeric_features = num_feat                      # ['year', 'mileage', ...]
categorical_features = cat_feat                  # ['Brand', 'model', 'transmission', 'fuelType']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)


X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In [8]:
# ---------------------------------------------------------
# BLENDING COM MINI FEATURE ENGINEERING
# ---------------------------------------------------------
import time
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error

# ---------------------------------------------------------
# 1. CONFIGURAÇÃO CROSS-VALIDATION
# ---------------------------------------------------------
N_SPLITS = 8
RANDOM_STATE = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]
DROP_FROM_MODEL = ["previousOwners"]

param_distributions = {
    "rf_n_estimators": [200],
    "rf_max_depth": [20],
    "rf_min_samples_leaf": [1],
    "rf_max_features": [0.5],
    "rf_bootstrap": [True],
    
    "et_n_estimators": [1200],
    "et_max_depth": [30],
    "et_min_samples_split": [4],
    "et_min_samples_leaf": [1],
    "et_max_features": [0.5],
    "et_bootstrap": [False],

    "hgb_max_iter": [1200],
    "hgb_learning_rate": [0.07],
    "hgb_max_depth": [20],
    "hgb_max_leaf_nodes": [191],
    "hgb_min_samples_leaf": [16],
    "hgb_l2_reg": [3.0],
    "hgb_loss": ["squared_error"]
}

N_RANDOM_CONFIGS = 10
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = []
best_rmse = np.inf
best_config = None

log_dir = "blending_logs"
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, "blending_with_age_log.txt")

with open(log_path, "w", encoding="utf-8") as log_file:
    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# START BLENDING WITH MINI FEATURE ENGINEERING")
    log(f"# TOTAL CONFIGS: {N_RANDOM_CONFIGS}")
    log("# =============================")

    for config_id, params in enumerate(sampler, start=1):
        config_start_time = time.time()
        log(f"\n######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        fold_rmses_val = []
        fold_rmses_train = []
        fold_maes_val = []
        fold_med_ae_val = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            fold_start_time = time.time()
            log(f"\n[C{config_id}|F{fold}] Processing fold...")

            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
            y_train_log = np.log1p(y_train)

            # -------------------------
            # 1) Feature Engineering & Imputation
            # -------------------------
            year_state = fit_year_median(X_train, "year", "model")
            X_train = transform_year_with_model_median(X_train, year_state)
            X_val   = transform_year_with_model_median(X_val, year_state)

            mileage_state = fit_mileage_imputer(X_train, "mileage", True)
            X_train = transform_mileage_imputer(X_train, mileage_state)
            X_val   = transform_mileage_imputer(X_val, mileage_state)

            engine_state = fit_engine_size_imputer(X_train, "engineSize")
            X_train = transform_engine_size_imputer(X_train, engine_state)
            X_val   = transform_engine_size_imputer(X_val, engine_state)

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, "mpg", True)
            X_train = transform_mpg_imputer(X_train, mpg_state)
            X_val   = transform_mpg_imputer(X_val, mpg_state)

            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)

            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

            # Age feature
            X_train = create_age_and_drop_year(X_train, "year", 2020)
            X_val   = create_age_and_drop_year(X_val, "year", 2020)

            # Drop previousOwners
            X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
            X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

            # -------------------------
            # 2) Encoding categorical features
            # -------------------------
            high_card = ["Brand", "model"]
            low_card  = [c for c in categorical_features if c not in high_card]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card], y_train_log)
            X_train_high = te.transform(X_train[high_card])
            X_val_high   = te.transform(X_val[high_card])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card])
            X_train_low = ohe.transform(X_train[low_card])
            X_val_low   = ohe.transform(X_val[low_card])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high, X_val_low], axis=1)

            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)
            X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

            # -------------------------
            # 3) Base Learners
            # -------------------------
            rf = RandomForestRegressor(
                n_estimators=params["rf_n_estimators"],
                max_depth=params["rf_max_depth"],
                min_samples_leaf=params["rf_min_samples_leaf"],
                max_features=params["rf_max_features"],
                bootstrap=params["rf_bootstrap"],
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            et = ExtraTreesRegressor(
                n_estimators=params["et_n_estimators"],
                max_depth=params["et_max_depth"],
                min_samples_leaf=params["et_min_samples_leaf"],
                max_features=params["et_max_features"],
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            hgb = HistGradientBoostingRegressor(
                max_iter=params["hgb_max_iter"],
                learning_rate=params["hgb_learning_rate"],
                max_depth=params["hgb_max_depth"],
                max_leaf_nodes=params["hgb_max_leaf_nodes"],
                min_samples_leaf=params["hgb_min_samples_leaf"],
                l2_regularization=params["hgb_l2_reg"],
                random_state=RANDOM_STATE
            )

            # Fit base learners individually
            rf.fit(X_train_final, y_train_log)
            et.fit(X_train_final, y_train_log)
            hgb.fit(X_train_final, y_train_log)

            # -------------------------
            # 4) Predictions for blender
            # -------------------------
            X_blend_train = pd.DataFrame({
                "rf": rf.predict(X_train_final),
                "et": et.predict(X_train_final),
                "hgb": hgb.predict(X_train_final)
            })
            X_blend_val = pd.DataFrame({
                "rf": rf.predict(X_val_final),
                "et": et.predict(X_val_final),
                "hgb": hgb.predict(X_val_final)
            })

            # -------------------------
            # 5) Train blender (meta-model)
            # -------------------------
            blender = LinearRegression()
            blender.fit(X_blend_train, y_train_log)

            # Predict with blender
            y_pred_train_log = blender.predict(X_blend_train)
            y_pred_val_log   = blender.predict(X_blend_val)

            y_pred_train = np.expm1(y_pred_train_log)
            y_pred_val   = np.expm1(y_pred_val_log)

            rmse_tr  = np.sqrt(mean_squared_error(y_train, y_pred_train))
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            med_ae_val = median_absolute_error(y_val, y_pred_val)

            fold_rmses_train.append(rmse_tr)
            fold_rmses_val.append(rmse_val)
            fold_maes_val.append(mae_val)
            fold_med_ae_val.append(med_ae_val)

            fold_time = time.time() - fold_start_time
            log(f"  > Fold {fold} | TRAIN_RMSE={rmse_tr:.2f} | VAL_RMSE={rmse_val:.2f} | MAE={mae_val:.2f} | Time={fold_time:.2f}s")

        # -------------------------
        # 6) Média folds
        # -------------------------
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_rmse_tr  = np.mean(fold_rmses_train)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_med_ae_val = np.mean(fold_med_ae_val)

        log(f"\nCONFIG {config_id} SUMMARY:")
        log(f"  RMSE_val={mean_rmse_val:.2f} | RMSE_train={mean_rmse_tr:.2f} | MAE_val={mean_mae_val:.2f} | MedAE_val={mean_med_ae_val:.2f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_val_mean": mean_rmse_val,
            "rmse_train_mean": mean_rmse_tr,
            "mae_val_mean": mean_mae_val,
            "medae_val_mean": mean_med_ae_val
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = params.copy()
            log(f"[NEW BEST RMSE] Config {config_id}")

# -------------------------
# 7) Resultados
# -------------------------
results_df = pd.DataFrame(search_results).sort_values("rmse_val_mean")
print("\nTop 5 Configs:")
display(results_df.head(5))

print("\nBest Config Found:")
print(best_config)


# =============================
# START BLENDING WITH MINI FEATURE ENGINEERING
# TOTAL CONFIGS: 10
# =============================

######## CONFIG 1/10 ########
Parameters: {'rf_n_estimators': 200, 'rf_min_samples_leaf': 1, 'rf_max_features': 0.5, 'rf_max_depth': 20, 'rf_bootstrap': True, 'hgb_min_samples_leaf': 16, 'hgb_max_leaf_nodes': 191, 'hgb_max_iter': 1200, 'hgb_max_depth': 20, 'hgb_loss': 'squared_error', 'hgb_learning_rate': 0.07, 'hgb_l2_reg': 3.0, 'et_n_estimators': 1200, 'et_min_samples_split': 4, 'et_min_samples_leaf': 1, 'et_max_features': 0.5, 'et_max_depth': 30, 'et_bootstrap': False}

[C1|F1] Processing fold...


c:\Users\Rosa Melo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


NameError: name 'LinearRegression' is not defined

In [5]:
# ---------------------------------------------------------
# STACKING COM MINI FEATURE ENGINEERING
# ---------------------------------------------------------
import time
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, r2_score

# ---------------------------------------------------------
# 1. CONFIGURAÇÃO CROSS-VALIDATION
# ---------------------------------------------------------
N_SPLITS = 8
RANDOM_STATE = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# features numéricas usadas (year → age; previousOwners excluído)
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]
DROP_FROM_MODEL = ["previousOwners"]

# hyperparameter space para random search
param_distributions = {
    "rf_n_estimators": [1000],
    "rf_max_depth": [20],
    "rf_min_samples_split": [2],
    "rf_min_samples_leaf": [1],
    "rf_max_features": [None],
    "rf_bootstrap": [True],

    "et_n_estimators": [1200],
    "et_max_depth": [30],
    "et_min_samples_split": [4],
    "et_min_samples_leaf": [1],
    "et_max_features": [0.5],
    'et_bootstrap': [False],

    "hgb_max_iter": [1200],
    "hgb_learning_rate": [0.07],
    "hgb_max_depth": [20],
    "hgb_max_leaf_nodes": [191],
    "hgb_min_samples_leaf": [16],
    "hgb_l2_reg": [3.0],
    "hgb_loss": ["squared_error"]
}

N_RANDOM_CONFIGS = 10
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = []
best_rmse = np.inf
best_config = None

log_dir = "stacking_logs"
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, "stacking_with_age_log.txt")

with open(log_path, "w", encoding="utf-8") as log_file:
    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# START STACKING WITH MINI FEATURE ENGINEERING")
    log(f"# TOTAL CONFIGS: {N_RANDOM_CONFIGS}")
    log("# =============================")

    for config_id, params in enumerate(sampler, start=1):
        config_start_time = time.time()
        log(f"\n######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        fold_rmses_val = []
        fold_rmses_train = []
        fold_maes_val = []
        fold_med_ae_val = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            fold_start_time = time.time()
            log(f"\n[C{config_id}|F{fold}] Processing fold...")

            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

            # log-transform target
            y_train_log = np.log1p(y_train)

            # -------------------------
            # 1) Feature Engineering & Imputation
            # -------------------------
            year_state = fit_year_median(X_train, "year", "model")
            X_train = transform_year_with_model_median(X_train, year_state)
            X_val   = transform_year_with_model_median(X_val, year_state)

            mileage_state = fit_mileage_imputer(X_train, "mileage", True)
            X_train = transform_mileage_imputer(X_train, mileage_state)
            X_val   = transform_mileage_imputer(X_val, mileage_state)

            engine_state = fit_engine_size_imputer(X_train, "engineSize")
            X_train = transform_engine_size_imputer(X_train, engine_state)
            X_val   = transform_engine_size_imputer(X_val, engine_state)

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, "mpg", True)
            X_train = transform_mpg_imputer(X_train, mpg_state)
            X_val   = transform_mpg_imputer(X_val, mpg_state)

            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)

            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

            # Age feature
            X_train = create_age_and_drop_year(X_train, "year", 2020)
            X_val   = create_age_and_drop_year(X_val, "year", 2020)

            # Drop previousOwners
            X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
            X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

            # -------------------------
            # 2) Encoding categorical features
            # -------------------------
            high_card = ["Brand", "model"]
            low_card  = [c for c in categorical_features if c not in high_card]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card], y_train_log)
            X_train_high = te.transform(X_train[high_card])
            X_val_high   = te.transform(X_val[high_card])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card])
            X_train_low = ohe.transform(X_train[low_card])
            X_val_low   = ohe.transform(X_val[low_card])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high, X_val_low], axis=1)

            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

            X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)
            
            scaler = StandardScaler()
            X_train_final = pd.DataFrame(scaler.fit_transform(X_train_final),
                                        columns=X_train_final.columns,
                                        index=X_train_final.index)
            X_val_final = pd.DataFrame(scaler.transform(X_val_final),
                                    columns=X_val_final.columns,
                                    index=X_val_final.index)


            # -------------------------
            # 3) Base Learners
            # -------------------------
            estimators = []

            rf = RandomForestRegressor(
                n_estimators=params["rf_n_estimators"],
                max_depth=params["rf_max_depth"],
                min_samples_split=params["rf_min_samples_split"],
                min_samples_leaf=params["rf_min_samples_leaf"],
                max_features=params["rf_max_features"],
                bootstrap=params["rf_bootstrap"],
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            estimators.append(("rf", rf))

            et = ExtraTreesRegressor(
                n_estimators=params["et_n_estimators"],
                max_depth=params["et_max_depth"],
                min_samples_leaf=params["et_min_samples_leaf"],
                max_features=params["et_max_features"],
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            estimators.append(("et", et))

            hgb = HistGradientBoostingRegressor(
                max_iter=params["hgb_max_iter"],
                learning_rate=params["hgb_learning_rate"],
                max_depth=params["hgb_max_depth"],
                max_leaf_nodes=params["hgb_max_leaf_nodes"],
                min_samples_leaf=params["hgb_min_samples_leaf"],
                l2_regularization=params["hgb_l2_reg"],
                random_state=RANDOM_STATE
            )
            estimators.append(("hgb", hgb))

            # -------------------------
            # 4) Stacking Regressor
            # -------------------------
            meta = Ridge(alpha= np.float64(4.0554607358408274))
            stack = StackingRegressor(
                estimators=estimators,
                final_estimator=meta,
                cv=5,
                n_jobs=1,
                passthrough=False
            )

            # -------------------------
            # 5) Treino e predição
            # -------------------------
            stack.fit(X_train_final, y_train_log)

            y_pred_train_log = stack.predict(X_train_final)
            y_pred_val_log   = stack.predict(X_val_final)

            y_pred_train = np.expm1(y_pred_train_log)
            y_pred_val   = np.expm1(y_pred_val_log)


            rmse_tr  = np.sqrt(mean_squared_error(y_train, y_pred_train))
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            med_ae_val = median_absolute_error(y_val, y_pred_val)
            fold_rmses_train.append(rmse_tr)
            fold_rmses_val.append(rmse_val)
            fold_maes_val.append(mae_val)
            fold_med_ae_val.append(med_ae_val)

            fold_time = time.time() - fold_start_time
            log(f"  > Fold {fold} | TRAIN_RMSE={rmse_tr:.2f} | VAL_RMSE={rmse_val:.2f} | MAE={mae_val:.2f} | Time={fold_time:.2f}s")

        # -------------------------
        # 6) Média folds
        # -------------------------
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_rmse_tr  = np.mean(fold_rmses_train)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_med_ae_val = np.mean(fold_med_ae_val)

        log(f"\nCONFIG {config_id} SUMMARY:")
        log(f"  RMSE_val={mean_rmse_val:.2f} | RMSE_train={mean_rmse_tr:.2f} | MAE_val={mean_mae_val:.2f} | MedAE_val={mean_med_ae_val:.2f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_val_mean": mean_rmse_val,
            "rmse_train_mean": mean_rmse_tr,
            "mae_val_mean": mean_mae_val,
            "medae_val_mean": mean_med_ae_val
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = params.copy()
            log(f"[NEW BEST RMSE] Config {config_id}")

# -------------------------
# 7) Resultados
# -------------------------
results_df = pd.DataFrame(search_results).sort_values("rmse_val_mean")
print("\nTop 5 Configs:")
display(results_df.head(5))

print("\nBest Config Found:")
print(best_config)


# =============================
# START STACKING WITH MINI FEATURE ENGINEERING
# TOTAL CONFIGS: 10
# =============================

######## CONFIG 1/10 ########
Parameters: {'rf_n_estimators': 1000, 'rf_min_samples_split': 2, 'rf_min_samples_leaf': 1, 'rf_max_features': None, 'rf_max_depth': 20, 'rf_bootstrap': True, 'hgb_min_samples_leaf': 16, 'hgb_max_leaf_nodes': 191, 'hgb_max_iter': 1200, 'hgb_max_depth': 20, 'hgb_loss': 'squared_error', 'hgb_learning_rate': 0.07, 'hgb_l2_reg': 3.0, 'et_n_estimators': 1200, 'et_min_samples_split': 4, 'et_min_samples_leaf': 1, 'et_max_features': 0.5, 'et_max_depth': 30, 'et_bootstrap': False}

[C1|F1] Processing fold...


c:\Users\Rosa Melo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


  > Fold 1 | TRAIN_RMSE=1296.49 | VAL_RMSE=1894.94 | MAE=1203.34 | Time=331.10s

[C1|F2] Processing fold...
  > Fold 2 | TRAIN_RMSE=1228.76 | VAL_RMSE=2098.08 | MAE=1239.90 | Time=355.23s

[C1|F3] Processing fold...
  > Fold 3 | TRAIN_RMSE=1221.67 | VAL_RMSE=2158.94 | MAE=1234.41 | Time=357.27s

[C1|F4] Processing fold...
  > Fold 4 | TRAIN_RMSE=1152.73 | VAL_RMSE=2593.96 | MAE=1257.87 | Time=345.76s

[C1|F5] Processing fold...


MemoryError: could not allocate 8388608 bytes